# 02 — MSP-Podcast→HCUDB 4クラスdecoder学習・評価

検証済みframe cacheとmanifestだけを入力にし、MSP-Podcast学習、HCUDB継続学習、両データセットの追加学習前後評価を行います。IEMOCAPは今回の一括研究経路には含めません。実データ1 epoch疎通と正式実行は別の出力先を使い、どちらも既定で無効です。


In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ser_pipeline').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ser_pipeline.cache import validate_cache
from ser_pipeline.notebook_api import environment_summary
from ser_pipeline.study import DatasetArtifacts, require_formal_epochs, run_transfer_study
from ser_pipeline.training import TrainingConfig

STUDY_DATASETS = ('msp_podcast', 'hcudb1')
RUN_REAL_SMOKE = False
RUN_FORMAL_SEED_42 = False
RUN_FORMAL_SEEDS_43_44 = False
FORMAL_EPOCHS = None

# 実測・検証結果をユーザーが確認した後だけTrueにします。
CONFIRM_CACHE_VALIDATION = False
CONFIRM_BENCHMARK_AND_CAPACITY = False
CONFIRM_SMOKE_COMPLETED = False
CONFIRM_SEED_42_ARTIFACTS = False

ARTIFACT_DIR = PROJECT_ROOT / 'runs' / 'ser_decoder_study'


def load_study_artifacts():
    artifacts = {}
    missing = []
    for name in STUDY_DATASETS:
        manifest_key = f'SER_{name.upper()}_MANIFEST'
        cache_key = f'SER_{name.upper()}_CACHE'
        if not os.environ.get(manifest_key):
            missing.append(manifest_key)
        if not os.environ.get(cache_key):
            missing.append(cache_key)
        exclusion_contract_path = None
        if name == 'msp_podcast':
            exclusion_key = 'SER_MSP_PODCAST_EXCLUSION_CONTRACT'
            if not os.environ.get(exclusion_key):
                missing.append(exclusion_key)
            else:
                exclusion_contract_path = Path(os.environ[exclusion_key])
        if manifest_key not in missing and cache_key not in missing:
            artifacts[name] = DatasetArtifacts(
                manifest_path=Path(os.environ[manifest_key]),
                cache_root=Path(os.environ[cache_key]),
                exclusion_contract_path=exclusion_contract_path,
            )
    if missing:
        raise ValueError(f'Missing study artifact environment variables: {missing}')
    return artifacts


def validate_execution_gates():
    if not CONFIRM_CACHE_VALIDATION:
        raise RuntimeError('Confirm both MSP/HCUDB caches were completely validated')
    if not CONFIRM_BENCHMARK_AND_CAPACITY:
        raise RuntimeError('Confirm the one-item CPU benchmark and the +20% capacity gate')
    artifacts = load_study_artifacts()
    cache_validation = {
        name: validate_cache(current.cache_root, current.manifest_path)
        for name, current in artifacts.items()
    }
    return artifacts, cache_validation


## 1. cache-only実行環境


In [ ]:
environment_summary()


## 2. 実行設定の確認


In [ ]:
{
    'datasets': STUDY_DATASETS,
    'device': 'cpu',
    'run_real_smoke': RUN_REAL_SMOKE,
    'run_formal_seed_42': RUN_FORMAL_SEED_42,
    'run_formal_seeds_43_44': RUN_FORMAL_SEEDS_43_44,
    'formal_epochs': FORMAL_EPOCHS,
}


## 3. 実データ1 epoch疎通（正式集計外）

seed 42でMSP親学習→HCUDB継続学習→両データセットの前後評価を行います。出力は`smoke/`に隔離され、正式結果には混ぜません。


In [ ]:
if RUN_REAL_SMOKE:
    smoke_artifacts, smoke_cache_validation = validate_execution_gates()
    smoke_summary = run_transfer_study(
        smoke_artifacts,
        ARTIFACT_DIR / 'smoke',
        seeds=(42,),
        base_config=TrainingConfig(seed=42, device='cpu', epochs=1),
    )
else:
    smoke_summary = {'status': 'disabled_by_default', 'seed': 42, 'epochs': 1}
smoke_summary


## 4. 正式seed 42実行ゲート

1 epoch疎通の時間と履歴を確認して`FORMAL_EPOCHS`を正の整数に固定し、先にseed 42だけを実行します。未設定のまま実行すると拒否します。


In [ ]:
if RUN_FORMAL_SEED_42:
    if not CONFIRM_SMOKE_COMPLETED:
        raise RuntimeError('Confirm the real-data 1 epoch smoke run before formal seed 42')
    formal_epochs = require_formal_epochs(FORMAL_EPOCHS)
    formal_artifacts, formal_cache_validation = validate_execution_gates()
    formal_seed_42_summary = run_transfer_study(
        formal_artifacts,
        ARTIFACT_DIR / 'formal' / 'initial-seed-42',
        seeds=(42,),
        base_config=TrainingConfig(seed=42, device='cpu', epochs=formal_epochs),
    )
else:
    formal_seed_42_summary = {
        'status': 'disabled_by_default', 'seed': 42, 'formal_epochs': FORMAL_EPOCHS
    }
formal_seed_42_summary


## 5. 正式seed 43・44実行ゲート

seed 42のcheckpoint、評価signature、cache ID、設定値を確認した後だけ実行します。出力はseed 42の正式出力と分けて保存します。


In [ ]:
if RUN_FORMAL_SEEDS_43_44:
    if not CONFIRM_SEED_42_ARTIFACTS:
        raise RuntimeError('Confirm the formal seed 42 artifacts before seeds 43 and 44')
    formal_epochs = require_formal_epochs(FORMAL_EPOCHS)
    followup_artifacts, followup_cache_validation = validate_execution_gates()
    formal_followup_summary = run_transfer_study(
        followup_artifacts,
        ARTIFACT_DIR / 'formal' / 'followup-seeds-43-44',
        seeds=(43, 44),
        base_config=TrainingConfig(seed=43, device='cpu', epochs=formal_epochs),
    )
else:
    formal_followup_summary = {
        'status': 'disabled_by_default', 'seeds': [43, 44], 'formal_epochs': FORMAL_EPOCHS
    }
formal_followup_summary
